# Map From Map Data

This notebook demonstrates how the cleaned `map_data.csv` file can power the Support Center Map. It shows the map-ready subset, the color rules, the category filters, and the popup content that the frontend team should reproduce later in the app.

## 1. Load Libraries and Map Data

This block imports the notebook libraries and reads the final cleaned CSV. It gives a quick overview so teammates can verify that the exported file has the expected structure before trying to render the map.

Expected output: a loaded dataframe and a summary of the categories available for mapping.

In [ ]:
# load the cleaned map_data.csv file and preview the dataset.
from pathlib import Path

import folium
import pandas as pd
from IPython.display import display

PLACEHOLDER = 'To be added'

DATA_DIR = Path.cwd()
if DATA_DIR.name != 'map':
    DATA_DIR = Path.cwd() / 'data' / 'map'

MAP_DATA_PATH = DATA_DIR / 'map_data.csv'
map_data = pd.read_csv(MAP_DATA_PATH)

print('Map data path:', MAP_DATA_PATH)
print('Shape:', map_data.shape)
print('\nCategory counts:')
print(map_data['category'].value_counts())
print('\nSchema preview:')
display(map_data.head(10))

Map data path: /Users/datong/Documents/5120/Nurodiversity inclusive design/dysLe_TP10/data/map/map_data.csv
Shape: (160, 15)

Category counts:
category
Intervention      83
Assessment        65
Support Groups    12
Name: count, dtype: int64

Schema preview:


,name,category,service_kind,profession_type,address,suburb,state,postcode,latitude,longitude,phone,website,source,description,highlighted
0,24-7Medcare Psychology,Assessment,Medical/Psych,Psychologist,"Ground Floor,737 Burwood Road, Hawthorn, VIC, ...",Hawthorn,VIC,3122.0,-37.8238804,145.0488157,To be added,To be added,Healthdirect NHSD Services Directory 2025,Assessment support identified from Healthdirec...,No
1,AMS Psychology and Consulting Cragieburn,Assessment,Medical/Psych,Psychologist,"Level 1,31 Craigieburn Road, Craigieburn, VIC,...",Craigieburn,VIC,3064.0,-37.5990452,144.939224,To be added,To be added,Healthdirect NHSD Services Directory 2025,Assessment support identified from Healthdirec...,No
2,Adelaide Road Psychology,Assessment,Medical/Psych,Psychologist,"76 Adelaide Road, Gawler South, SA, 5118.0",Gawler South,SA,5118.0,-34.61116028,138.74368286,To be added,To be added,Healthdirect NHSD Services Directory 2025,Assessment support identified from Healthdirec...,No
3,Balanced Minds Psychology Practice,Assessment,Medical/Psych,Psychologist,"The Italian Forum,SUITE 47A,23 NORTON STREET, ...",Leichhardt,NSW,2040.0,-33.887211,151.1583777,To be added,To be added,Healthdirect NHSD Services Directory 2025,Assessment support identified from Healthdirec...,No
4,Bluebird Psychology,Assessment,Medical/Psych,Psychologist,"Unit 18,15-17 Terminus Street, Castle Hill, NS...",Castle Hill,NSW,2154.0,-33.73423767,151.00598145,To be added,To be added,Healthdirect NHSD Services Directory 2025,Assessment support identified from Healthdirec...,No
5,Central Psychology - Macedon Street Consulting...,Assessment,Medical/Psych,Psychologist,"9 MacEdon Street, Sunbury, VIC, 3429.0",Sunbury,VIC,3429.0,-37.57733917,144.73275757,To be added,To be added,Healthdirect NHSD Services Directory 2025,Assessment support identified from Healthdirec...,No
6,Child and Adolescent Psychology Services,Assessment,Medical/Psych,Psychologist,"164 THE PARADE, Ascot Vale, VIC, 3032.0",Ascot Vale,VIC,3032.0,-37.77180225,144.90969008,To be added,To be added,Healthdirect NHSD Services Directory 2025,Assessment support identified from Healthdirec...,No
7,Clear Health Psychology Alkimos,Assessment,Medical/Psych,Psychologist,"Unit 1 & 2,15 Graceful Boulevard, Alkimos, WA,...",Alkimos,WA,6038.0,-31.6171563,115.6805868,To be added,To be added,Healthdirect NHSD Services Directory 2025,Assessment support identified from Healthdirec...,No
8,Cocoon Psychology,Assessment,Medical/Psych,Psychologist,"Level 2,8 KEILOR ROAD, Essendon North, VIC, 30...",Essendon North,VIC,3041.0,-37.74394247,144.90915579,To be added,To be added,Healthdirect NHSD Services Directory 2025,Assessment support identified from Healthdirec...,No
9,Cognition Psychology,Assessment,Medical/Psych,Psychologist,"55 Painswick Street, Berserker, QLD, 4701.0",Berserker,QLD,4701.0,-23.36768913,150.52250671,To be added,To be added,Healthdirect NHSD Services Directory 2025,Assessment support identified from Healthdirec...,No


## 2. Prepare Map-Renderable Records

This block separates rows that can become map pins from rows that still lack valid coordinates. This distinction is important because address-only rows are still useful in the dataset, but they should not be rendered as pins until coordinates exist.

Expected output: two dataframes, one renderable and one unresolved.

In [ ]:
# split rows into rows that can be drawn as pins and rows still missing coordinates.
renderable = map_data[(map_data['latitude'] != PLACEHOLDER) & (map_data['longitude'] != PLACEHOLDER)].copy()
unresolved = map_data[(map_data['latitude'] == PLACEHOLDER) | (map_data['longitude'] == PLACEHOLDER)].copy()

renderable['latitude'] = renderable['latitude'].astype(float)
renderable['longitude'] = renderable['longitude'].astype(float)

print('Renderable rows:', len(renderable))
print('Rows still missing coordinates:', len(unresolved))
display(unresolved[['name', 'address', 'website']].head(20))

Renderable rows: 157
Rows still missing coordinates: 3


,name,address,website
149,Australian Dyslexia Foundation,To be added,To be added
156,Speld Sa Inc,"U 2 259 Glen Osmond Rd, Frewville, SA, 5063.0",http://www.speldsa.org.au
158,Square Pegs Dyslexia Support and Advocacy Inc,"2 Hargrave Pl, Mount Nelson, TAS, 7007.0",https://www.squarepegstas.org


## 3. Define Visual Rules

This block defines the visual language for the notebook map and for the later frontend handoff. The three service kinds map directly to the three pin colors agreed for the project.

Expected output: a simple color mapping and the approved category filter list.

In [ ]:
# define the agreed pin colors and category filter order.
COLOR_MAP = {
    'Medical/Psych': 'blue',
    'Education': 'green',
    'Community': 'orange',
}

CATEGORY_ORDER = ['Assessment', 'Intervention', 'Support Groups']

print('Color map:', COLOR_MAP)
print('Category filters:', CATEGORY_ORDER)

Color map: {'Medical/Psych': 'blue', 'Education': 'green', 'Community': 'orange'}
Category filters: ['Assessment', 'Intervention', 'Support Groups']


## 4. Create a Reusable Map Function

This block creates a helper function for rendering a Folium map from a filtered dataframe. Each popup shows the minimum agreed information: name, profession type, contact information, and description.

Expected output: a reusable function for the all-records map and each category-specific map.

In [ ]:
# define reusable functions for popup text and Folium map creation.
def popup_html(row):
    # This function controls what users see after clicking a pin.
    # Use HTML line breaks because Folium popups render HTML text.
    return f"""
    <div style='min-width: 220px;'>
      <strong>{row['name']}</strong><br>
      Profession Type: {row['profession_type']}<br>
      Address: {row['address']}<br>
      Phone: {row['phone']}<br>
      Website: {row['website']}<br>
      Description: {row['description']}<br>
      Highlighted: {row['highlighted']}
    </div>
    """


def make_map(df, title):
    # Create a base map centered on Australia.
    # The zoom level is low so users can see the whole country first.
    fmap = folium.Map(location=[-28.0, 134.0], zoom_start=4, tiles='OpenStreetMap')
    # Add one marker for each row in the filtered dataset.
    for _, row in df.iterrows():
        # Choose marker color based on service type.
        icon_color = COLOR_MAP.get(row['service_kind'], 'gray')
        # Highlight trusted organisations with a star icon.
        icon_name = 'star' if row['highlighted'] == 'Yes' else 'info-sign'
        # Create the marker and attach the popup text.
        folium.Marker(
            location=[row['latitude'], row['longitude']],
            popup=folium.Popup(popup_html(row), max_width=320),
            tooltip=row['name'],
            icon=folium.Icon(color=icon_color, icon=icon_name),
        ).add_to(fmap)
    display(title)
    return fmap

## 5. Render the Base Map

This block renders the full Australia-wide map using all rows that already have valid coordinates. It shows what the complete support-center experience looks like before category filtering is applied.

Expected output: one base map with all available pins.

In [ ]:
# Render the full map using every row with valid coordinates.
make_map(renderable, 'All support locations')

'All support locations'

## 6. Demonstrate Category Filtering

This block shows one filtered map for each approved category. This is the notebook equivalent of the category filter chips that the frontend team will later implement in the app.

Expected output: three category-specific maps.

In [ ]:
# render one map per category to demonstrate filtering.
for category in CATEGORY_ORDER:
    filtered = renderable[renderable['category'] == category].copy()
    print(f'{category}:', len(filtered))
    display(make_map(filtered, f'{category} locations'))

Assessment: 65


'Assessment locations'

Intervention: 83


'Intervention locations'

Support Groups: 9


'Support Groups locations'

## 7. Highlight Trusted Organisations

This block previews the trusted organizations flagged with `highlighted = Yes`. These entries should receive extra visual emphasis in the final frontend map because they represent well-known support organizations such as SPELD, DSF, and AUSPELD.

Expected output: a table of highlighted rows and a matching highlighted-only map view.

In [7]:
# Step 7 code note: show trusted/highlighted organisations separately.
highlighted = renderable[renderable['highlighted'] == 'Yes'].copy()
display(highlighted[['name', 'category', 'service_kind', 'website']])
make_map(highlighted, 'Highlighted organisations')

,name,category,service_kind,website
148,Auspeld,Support Groups,Community,https://www.auspeld.org.au
153,Speld Foundation Of South Australia,Support Groups,Community,http://www.speldsa.org.au
154,Speld NSW Inc,Support Groups,Community,http://www.speldnsw.org.au
155,Speld Qld Inc.,Support Groups,Community,https://www.speld.org.au
157,Speld-Victoria Inc,Support Groups,Community,https://www.speldvic.org.au
159,The Dyslexia Speld Foundation Wa Inc,Support Groups,Community,https://www.dsf.net.au


'Highlighted organisations'

## 8. Frontend Handoff Notes

The final app implementation should mirror these rules:
- Use suburb/state text search rather than collecting live user location.
- Use `Category` to filter by `Assessment`, `Intervention`, and `Support Groups`.
- Color pins by `service_kind`: blue for `Medical/Psych`, green for `Education`, yellow-style community color for `Community`.
- Show popup fields from `map_data.csv`.
- Do not render rows without coordinates as pins.

Expected output: a concise checklist teammates can reuse when they build the production map.

In [8]:
# Step 8 code note: print the handoff checklist for future frontend implementation.
handoff_columns = [
    'name', 'category', 'service_kind', 'profession_type', 'address', 'suburb', 'state',
    'postcode', 'latitude', 'longitude', 'phone', 'website', 'source', 'description', 'highlighted'
]
print('Frontend handoff columns:')
for column in handoff_columns:
    print('-', column)

Frontend handoff columns:
- name
- category
- service_kind
- profession_type
- address
- suburb
- state
- postcode
- latitude
- longitude
- phone
- website
- source
- description
- highlighted
